# Sparkle circuits with Mermaid diagrams

This notebook demonstrates how to author Sparkle hardware modules and render their structural block diagrams as Mermaid `flowchart LR` graphs in xeus-lean.

## Two ways to render Mermaid in this kernel

1. **Markdown cell with a Mermaid code-fence** (Jupyter Lab 4+ / Notebook 7+ render this natively).
2. **`#mermaid "..."` command** from `TutorialExtended.MermaidHelper` — works everywhere, dynamically generates the diagram from Lean code.

Path 1 is simpler for fixed diagrams; path 2 is for diagrams generated at runtime.

## Path 1: Markdown cell rendering

```mermaid
flowchart LR
  en([en : Bool]) --> Counter[Counter<br/>BitVec 8]
  Counter --> Out([count : BitVec 8])
```

## Path 2: `#mermaid` command from a Lean cell

In [ ]:
import TutorialExtended.MermaidHelper
open TutorialExtended.MermaidHelper

#mermaid "flowchart LR
  en([en : Bool]) --> C[Counter]
  C --> P[ParityCheck]
  C --> M[Monitor]
  P --> Alert([parityAlert])
  M --> Hit([thresholdHit])"

## A real Sparkle module: 8-bit counter

Define a counter, evaluate it, then render the architecture diagram.

In [ ]:
import Sparkle

open Sparkle.Core.Domain
open Sparkle.Core.Signal

def counter8 {dom : DomainConfig}
    (en : Signal dom Bool) : Signal dom (BitVec 8) :=
  Signal.loop fun count =>
    let next := Signal.mux en (count + 1#8) count
    Signal.register 0#8 next

#eval (counter8 (dom := defaultDomain) (Signal.pure true)).sample 8

In [ ]:
#mermaid "flowchart LR
  en([en : Bool]) --> Mux[Signal.mux]
  count([count]) --> AddOne[count + 1]
  AddOne --> Mux
  count --> Mux
  Mux --> Reg[Signal.register 0]
  Reg -- count : BitVec 8 --> count"

## LTL bug-localization (Step 6 of the LTL tutorial)

The 3-premise contract for a write→hold→read register:

In [ ]:
#mermaid "flowchart LR
  writeEn([writeEn]) --> P1
  data([data]) --> P1
  P1[P1: write commits<br/>cycle-N+1 update] --> reg[(reg)]
  reg --> P2[P2: no-write<br/>preserves]
  P2 --> reg
  reg --> P3[P3: read observes<br/>current reg]
  readReady([readReady]) --> P3
  P3 --> observed([observed])"

## Generating diagrams from circuit data

Because `#mermaid` takes a `String`, you can build it from Lean code — for instance, list out a register's wireNames and emit one block per field:

In [ ]:
import TutorialExtended.MermaidHelper
open TutorialExtended.MermaidHelper

/-- Generate a flowchart LR with N parallel branches from a list. -/
def fanoutDiagram (src : String) (dsts : List String) : String :=
  let edges := dsts.foldl (fun acc d => acc ++ s!"\n  {src} --> {d}") "flowchart LR"
  edges

#eval mermaid (fanoutDiagram "Counter" ["Monitor", "ParityCheck", "Logger"])